# 040 — Inference & Delta Analysis

Load the best model and run batch predictions on all images in `data/test/rgb/`.
Predictions are saved to `data/test/predicted/` with the architecture name embedded
in the filename: `{stem}_{arch}_predicted_ir.jpg`.

When a matching IR image exists in `data/test/ir/` (same filename), the notebook also
computes and visualises the delta `|real_IR − predicted_IR|` for underdrawing inspection.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from skimage import exposure

from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.inference_utils import predict_with_overlap
from scripts.trainer import load_model
from scripts.visualization import plot_delta, plot_predictions

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. Load model

Change `ARCH` to switch between architectures:
`"unet"` | `"resunet"` | `"attention_unet"` | `"efficientnet_unet"`

`ADVANCED_LOSS` is derived automatically from `ARCH` — it must match the loss
used at training time for correct metric computation.

In [ ]:
ARCH = "efficientnet_unet"  # "unet" | "resunet" | "attention_unet" | "efficientnet_unet"

model = load_model(
    ARCH,
    model_dir=settings.MODELS_DIR,
    lr=settings.LEARNING_RATE,
    loss_alpha=settings.LOSS_ALPHA,
)
print(f"Loaded: {model.name} | Parameters: {model.count_params():,}")

## 2. Test data setup

Images are read from:
- `data/test/rgb/` — RGB input images
- `data/test/ir/`  — real IR counterparts *(optional — enables delta analysis)*

Predictions are saved to:
- `data/test/predicted/{stem}_{arch}_predicted_ir.jpg`

If an image in `data/test/rgb/` has no matching file in `data/test/ir/`,
inference still runs but delta visualisation is skipped for that image.

In [ ]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR  = project_root / "data" / "test" / "ir"
PRED_DIR     = project_root / "data" / "test" / "predicted"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# All RGB inputs, sorted
rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg"))
print(f"RGB images:  {len(rgb_paths)}")

# Pair each RGB with the IR of the same name (if present)
test_pairs = [(p, TEST_IR_DIR / p.name) for p in rgb_paths if (TEST_IR_DIR / p.name).exists()]
rgb_only   = [p for p in rgb_paths if not (TEST_IR_DIR / p.name).exists()]
print(f"Paired:      {len(test_pairs)}  (RGB + IR available)")
print(f"RGB-only:    {len(rgb_only)}   (no matching IR — predictions only)")


def predict_and_save(
    model: tf.keras.Model,
    rgb_path: Path,
    arch: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Load an RGB image, predict IR, save to ``PRED_DIR``, return arrays.

    Parameters
    ----------
    model : tf.keras.Model
        Trained IR prediction model.
    rgb_path : Path
        Path to the input RGB image.
    arch : str
        Architecture name embedded in the output filename.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        ``(rgb, ir_pred)`` normalised to ``[0, 1]``.
        ``rgb`` shape ``(H, W, 3)``, ``ir_pred`` shape ``(H, W, 1)``.
    """
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    h, w = rgb.shape[:2]

    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    ir_pred = model.predict(padded[tf.newaxis, ...], verbose=0)[0]
    ir_pred = ir_pred[:h, :w, :]

    out_path = PRED_DIR / f"{rgb_path.stem}_{arch}_predicted_ir.jpg"
    Image.fromarray(
        (ir_pred.squeeze() * 255).clip(0, 255).astype(np.uint8)
    ).save(out_path)

    return rgb, ir_pred

## 3. Batch predictions

Run inference on every image in `data/test/rgb/` and save results to
`data/test/predicted/`. Predictions are cached in memory for the
visualisation steps that follow.

In [ ]:
print(f"Running '{ARCH}' on {len(rgb_paths)} image(s)...")
predictions: dict[str, tuple[np.ndarray, np.ndarray]] = {}

for rgb_path in rgb_paths:
    rgb, ir_pred = predict_and_save(model, rgb_path, ARCH)
    predictions[rgb_path.stem] = (rgb, ir_pred)
    print(f"  saved: {rgb_path.stem}_{ARCH}_predicted_ir.jpg")

print(f"\nDone. {len(predictions)} prediction(s) saved to:\n  {PRED_DIR}")

## 4. Visual comparison — paired images

RGB input, real IR, and predicted IR displayed side by side for each
image that has a matching counterpart in `data/test/ir/`.

In [ ]:
N_DISPLAY = min(5, len(test_pairs))

for rgb_path, ir_path in test_pairs[:N_DISPLAY]:
    rgb, ir_pred = predictions[rgb_path.stem]
    ir_real = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    ir_real = ir_real[..., np.newaxis]

    fig = plot_predictions(rgb, ir_real, ir_pred, title=rgb_path.stem)
    plt.show()

## 5. Delta heatmap

`delta = |real_IR − predicted_IR|` for each paired image.
High-delta regions indicate where the model's expectation diverges from the observed IR
— a strong indicator of hidden underdrawings.

In [ ]:
for i, (rgb_path, ir_path) in enumerate(test_pairs):
    rgb, ir_pred = predictions[rgb_path.stem]
    ir_real = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    ir_real = ir_real[..., np.newaxis]

    delta = np.abs(ir_real.squeeze() - ir_pred.squeeze())

    # Save raw delta
    out_path = PRED_DIR / f"{rgb_path.stem}_{ARCH}_delta.jpg"
    Image.fromarray((delta * 255).clip(0, 255).astype(np.uint8)).save(out_path)

    if i < N_DISPLAY:
        fig = plot_delta(ir_real, ir_pred, title=f"Delta — {rgb_path.stem}")
        plt.show()

print(f"{len(test_pairs)} delta(s) saved to {PRED_DIR}")

## 6. CLAHE-enhanced delta for underdrawing inspection

Contrast Limited Adaptive Histogram Equalisation (CLAHE) amplifies local
contrast in the delta map, making faint underdrawing traces more visible.

In [ ]:
CLAHE_CLIP = 0.03  # Increase for more aggressive contrast enhancement

for i, (rgb_path, ir_path) in enumerate(test_pairs):
    rgb, ir_pred = predictions[rgb_path.stem]
    ir_real = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    ir_real = ir_real[..., np.newaxis]

    delta       = np.abs(ir_real.squeeze() - ir_pred.squeeze())
    delta_clahe = exposure.equalize_adapthist(delta, clip_limit=CLAHE_CLIP)

    # Save CLAHE-enhanced delta
    out_path = PRED_DIR / f"{rgb_path.stem}_{ARCH}_delta_clahe.jpg"
    Image.fromarray((delta_clahe * 255).clip(0, 255).astype(np.uint8)).save(out_path)

    if i < 3:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"CLAHE underdrawing detection — {rgb_path.stem}")

        axes[0].imshow(rgb);                            axes[0].set_title("RGB");         axes[0].axis("off")
        axes[1].imshow(ir_real.squeeze(), cmap="gray"); axes[1].set_title("Real IR");     axes[1].axis("off")
        im_raw   = axes[2].imshow(delta,       cmap="hot"); axes[2].set_title("Raw delta");   axes[2].axis("off")
        im_clahe = axes[3].imshow(delta_clahe, cmap="hot"); axes[3].set_title("CLAHE delta"); axes[3].axis("off")
        plt.colorbar(im_raw,   ax=axes[2], fraction=0.046, pad=0.04)
        plt.colorbar(im_clahe, ax=axes[3], fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()

print(f"{len(test_pairs)} CLAHE delta(s) saved to {PRED_DIR}")

## 7. Overlap-based inference with Gaussian blending

`predict_with_overlap` splits any input image into overlapping square patches, runs the
model on each, and blends predictions with per-pixel Gaussian weights. This eliminates
the hard seam artifacts that appear with non-overlapping stitching, and is particularly
useful for images larger than the training patch size.

`stride` controls the step between consecutive patches; the default `stride = patch_size // 2`
(50 % overlap) gives smooth blending with manageable compute cost.

In [ ]:
if not test_pairs:
    print("No paired images available — add IR files to data/test/ir/ to enable this section.")
else:
    rgb_path, ir_path = test_pairs[0]

    rgb_full = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir_real  = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    ir_real  = ir_real[..., np.newaxis]

    ir_overlap = predict_with_overlap(model, rgb_full, patch_size=256, stride=128)

    out_path = PRED_DIR / f"{rgb_path.stem}_{ARCH}_overlap_predicted_ir.jpg"
    Image.fromarray(
        (ir_overlap.squeeze() * 255).clip(0, 255).astype(np.uint8)
    ).save(out_path)
    print(f"Overlap prediction saved: {out_path.name}")

    fig = plot_predictions(
        rgb_full, ir_real, ir_overlap,
        title=f"Overlap inference (patch=256, stride=128) — {rgb_path.stem}",
    )
    plt.show()

    fig = plot_delta(ir_real, ir_overlap, title=f"Delta (overlap) — {rgb_path.stem}")
    plt.show()